# Phase 3 — Meta-Feature Extraction

Extracts three meta-feature representations for every dataset in `meta_training.csv`:

| Option | Representation | Dim | Role |
|--------|---------------|-----|------|
| A | Hand-crafted (pymfe-style) | ~20 | Main approach |
| B | Autoencoder bottleneck | 2K | Ablation 1 |
| C | Dictionary Learning sparse codes | 2K | Ablation 2 — novel contribution |

**Outputs**:
- `data/meta_table/meta_training.csv` — updated with Option A columns (main table)
- `data/meta_table/meta_training_optA.csv` — Option A features + LSE targets
- `data/meta_table/meta_training_optB.csv` — Option B features + LSE targets
- `data/meta_table/meta_training_optC.csv` — Option C features + LSE targets

**Rules**: same random_state=42 throughout; showcase datasets never touched.

In [1]:
import os, sys, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Add src/ to path
def find_project_root(start=None):
    cur = Path(start or os.getcwd()).resolve()
    for path in (cur, *cur.parents):
        if (path / 'CLAUDE.md').exists() and (path / 'src').exists():
            return str(path)
    raise RuntimeError('Project root not found; run from the repo root or notebooks folder')

ROOT = find_project_root()
if os.path.join(ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, 'src'))

from metafeatures import extract_optA, extract_optB, extract_optC

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DIR   = os.path.join(ROOT, 'data', 'raw')
META_DIR  = os.path.join(ROOT, 'data', 'meta_table')
LSE_CSV   = os.path.join(META_DIR, 'meta_training.csv')
MANIFEST  = os.path.join(META_DIR, 'dataset_manifest.csv')
CKPT_A    = os.path.join(META_DIR, 'mf_checkpoint_A.csv')
CKPT_B    = os.path.join(META_DIR, 'mf_checkpoint_B.csv')
CKPT_C    = os.path.join(META_DIR, 'mf_checkpoint_C.csv')
OUT_A     = os.path.join(META_DIR, 'meta_training_optA.csv')
OUT_B     = os.path.join(META_DIR, 'meta_training_optB.csv')
OUT_C     = os.path.join(META_DIR, 'meta_training_optC.csv')

openml.config.cache_directory = RAW_DIR

SEED = 42
np.random.seed(SEED)

SHOWCASE_IDS = {61, 187, 15, 53, 40966, 37, 54, 1590, 1597}

print('Paths OK')

Paths OK


In [2]:
# ── Load LSE table + showcase guard ───────────────────────────────────────────
lse_df = pd.read_csv(LSE_CSV)
leaked = set(lse_df['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak: {leaked}'

manifest = pd.read_csv(MANIFEST)
n_cls_map = dict(zip(manifest['dataset_id'], manifest['n_classes']))

LSE_COLS = ['LSE_kmeans', 'LSE_dbscan', 'LSE_agg', 'LSE_gmm', 'LSE_autoenc', 'LSE_dictlearn']
DATASET_IDS = lse_df['dataset_id'].tolist()

print(f'Datasets to process: {len(DATASET_IDS)}')
lse_df.head(3)

Datasets to process: 51


,dataset_id,LSE_kmeans,LSE_dbscan,LSE_agg,LSE_gmm,LSE_autoenc,LSE_dictlearn,best_method,gt_accuracy
0,43924,0.7846,0.7846,0.7846,0.7846,0.7846,0.8923,dictlearn,0.4392
1,732,0.6585,0.6585,0.6829,0.6585,0.5854,0.6585,agg,0.8200
2,1459,0.2468,0.2356,0.2425,0.2527,0.2756,0.2585,autoenc,0.9178


In [3]:
# ── Dataset loader (same logic as Phase 2) ────────────────────────────────────

def load_and_split(dataset_id):
    """Download (cached), encode labels, 80/20 stratified split."""
    ds = openml.datasets.get_dataset(
        dataset_id,
        download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )
    X = X.select_dtypes(include=[np.number]).astype(float)
    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))
    X_tr, X_te, y_tr, y_te = train_test_split(
        X.values, y_enc,
        test_size=0.2,
        random_state=SEED,
        stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te

## Option A — Hand-Crafted Meta-Features

In [4]:
# ── Resume from checkpoint ────────────────────────────────────────────────────
if os.path.exists(CKPT_A):
    ckpt_a = pd.read_csv(CKPT_A)
    done_a = set(ckpt_a['dataset_id'])
    rows_a = ckpt_a.to_dict('records')
    print(f'Resuming Option A — {len(done_a)} done')
else:
    done_a, rows_a = set(), []
    print('Starting Option A fresh')

total = len(DATASET_IDS)

for i, did in enumerate(DATASET_IDS):
    if did in done_a:
        continue

    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        feats = extract_optA(X_tr, y_tr, X_te, y_te)
        feats['dataset_id'] = did
        rows_a.append(feats)
        done_a.add(did)
        elapsed = time.time() - t0
        print(f'[{i+1:3d}/{total}]  did={did}  ok  ({elapsed:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_a.append({'dataset_id': did})
        done_a.add(did)

    pd.DataFrame(rows_a).to_csv(CKPT_A, index=False)

optA_df = pd.DataFrame(rows_a)
print(f'\nOption A done. Shape: {optA_df.shape}')
optA_df.head(3)

Starting Option A fresh
[  1/51]  did=43924  ok  (0.2s)
[  2/51]  did=732  ok  (2.7s)
[  3/51]  did=1459  ok  (0.2s)
[  4/51]  did=4538  ok  (0.4s)
[  5/51]  did=694  ok  (0.0s)
[  6/51]  did=43958  ok  (0.1s)
[  7/51]  did=42544  ok  (0.0s)
[  8/51]  did=46944  ok  (0.1s)
[  9/51]  did=1446  ok  (0.1s)
[ 10/51]  did=45035  ok  (1.0s)
[ 11/51]  did=39  ok  (0.0s)
[ 12/51]  did=44033  ok  (0.4s)
[ 13/51]  did=1527  ok  (0.1s)
[ 14/51]  did=41842  ok  (0.4s)
[ 15/51]  did=44367  ok  (0.1s)
[ 16/51]  did=41168  ok  (5.7s)
[ 17/51]  did=44505  ok  (0.2s)
[ 18/51]  did=45060  ok  (0.2s)
[ 19/51]  did=1513  ok  (0.0s)
[ 20/51]  did=46543  ok  (0.5s)
[ 21/51]  did=41858  ok  (0.1s)
[ 22/51]  did=45028  ok  (0.4s)
[ 23/51]  did=734  ok  (0.3s)
[ 24/51]  did=46532  ok  (0.1s)
[ 25/51]  did=46842  ok  (0.0s)
[ 26/51]  did=44528  ok  (0.1s)
[ 27/51]  did=46952  ok  (0.1s)
[ 28/51]  did=4154  ok  (0.2s)
[ 29/51]  did=784  ok  (0.0s)
[ 30/51]  did=44671  ok  (0.2s)
[ 31/51]  did=45714  ok  (0.3s)
[

,n_instances,n_features,n_classes,skewness_mean,kurtosis_mean,mean_abs_pearson,class_entropy,imbalance_ratio,hopkins,intrinsic_dim_ratio,pca_var_pc1,inter_intra_ratio,silhouette_true,davies_bouldin_true,knn1_accuracy,decision_stump_accuracy,feature_sparsity,cv_mean,dataset_id
0,588.0,5.0,5.0,2.329387,30.215164,0.137909,0.974066,2.035714,1.000000,1.000000,0.317917,0.815182,-0.057171,7.322711,0.350355,0.382689,0.000000,0.246955,43924
1,200.0,50.0,2.0,0.078808,1.183541,0.054789,0.997402,1.127660,0.725065,0.880000,0.041514,1.653333,0.008383,8.480350,0.550000,0.660000,0.005800,113.350427,732
2,8174.0,7.0,10.0,0.681386,0.471077,0.256039,0.988820,2.360417,0.999999,0.857143,0.328560,1.457884,-0.070544,7.303419,0.841445,0.181062,0.006956,0.806227,1459


## Option B — Autoencoder Bottleneck

In [5]:
# ── Resume from checkpoint ────────────────────────────────────────────────────
if os.path.exists(CKPT_B):
    ckpt_b = pd.read_csv(CKPT_B)
    done_b = set(ckpt_b['dataset_id'])
    rows_b = ckpt_b.to_dict('records')
    print(f'Resuming Option B — {len(done_b)} done')
else:
    done_b, rows_b = set(), []
    print('Starting Option B fresh')

for i, did in enumerate(DATASET_IDS):
    if did in done_b:
        continue

    n_cls = int(n_cls_map.get(did, 2))
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        vec = extract_optB(X_tr, n_cls)
        bottleneck_dim = len(vec) // 2
        rec = {'dataset_id': did}
        for j, v in enumerate(vec[:bottleneck_dim]):
            rec[f'ae_mean_{j}'] = float(v)
        for j, v in enumerate(vec[bottleneck_dim:]):
            rec[f'ae_var_{j}'] = float(v)
        rows_b.append(rec)
        done_b.add(did)
        elapsed = time.time() - t0
        print(f'[{i+1:3d}/{total}]  did={did}  bottleneck={bottleneck_dim}  ({elapsed:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_b.append({'dataset_id': did})
        done_b.add(did)

    pd.DataFrame(rows_b).to_csv(CKPT_B, index=False)

optB_df = pd.DataFrame(rows_b)
print(f'\nOption B done. Shape: {optB_df.shape}')
optB_df.head(3)

Starting Option B fresh
[  1/51]  did=43924  bottleneck=10  (1.7s)
[  2/51]  did=732  bottleneck=8  (0.1s)
[  3/51]  did=1459  bottleneck=20  (3.5s)
[  4/51]  did=4538  bottleneck=10  (3.5s)
[  5/51]  did=694  bottleneck=18  (0.1s)
[  6/51]  did=43958  bottleneck=8  (1.9s)
[  7/51]  did=42544  bottleneck=16  (0.1s)
[  8/51]  did=46944  bottleneck=8  (0.7s)
[  9/51]  did=1446  bottleneck=8  (0.1s)
[ 10/51]  did=45035  bottleneck=8  (3.5s)
[ 11/51]  did=39  bottleneck=16  (0.2s)
[ 12/51]  did=44033  bottleneck=8  (3.1s)
[ 13/51]  did=1527  bottleneck=10  (1.1s)
[ 14/51]  did=41842  bottleneck=8  (3.2s)
[ 15/51]  did=44367  bottleneck=8  (0.8s)
[ 16/51]  did=41168  bottleneck=8  (4.2s)
[ 17/51]  did=44505  bottleneck=14  (0.8s)
[ 18/51]  did=45060  bottleneck=8  (2.9s)
[ 19/51]  did=1513  bottleneck=10  (0.1s)
[ 20/51]  did=46543  bottleneck=8  (3.4s)
[ 21/51]  did=41858  bottleneck=8  (0.6s)
[ 22/51]  did=45028  bottleneck=8  (3.0s)
[ 23/51]  did=734  bottleneck=8  (3.7s)
[ 24/51]  did=4

,dataset_id,ae_mean_0,ae_mean_1,ae_mean_2,ae_mean_3,ae_mean_4,ae_mean_5,ae_mean_6,ae_mean_7,ae_mean_8,...,ae_var_10,ae_var_11,ae_var_12,ae_var_13,ae_var_14,ae_var_15,ae_var_16,ae_var_17,ae_var_18,ae_var_19
0,43924,0.388538,-0.489906,0.417240,-0.377791,1.012906,-0.351539,0.629131,1.525327,-0.425829,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,732,0.306123,-0.120100,-0.018270,0.097530,0.479096,0.540394,-0.109709,0.467770,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1459,0.400284,-0.252291,-0.138069,-1.301421,0.556215,0.000788,-0.101478,-0.829537,-0.227343,...,0.372076,0.994144,0.330773,0.728328,0.497839,0.436438,0.127562,0.355001,0.575314,0.629349


## Option C — Dictionary Learning Sparse Codes

In [6]:
# ── Resume from checkpoint ────────────────────────────────────────────────────
if os.path.exists(CKPT_C):
    ckpt_c = pd.read_csv(CKPT_C)
    done_c = set(ckpt_c['dataset_id'])
    rows_c = ckpt_c.to_dict('records')
    print(f'Resuming Option C — {len(done_c)} done')
else:
    done_c, rows_c = set(), []
    print('Starting Option C fresh')

for i, did in enumerate(DATASET_IDS):
    if did in done_c:
        continue

    n_cls = int(n_cls_map.get(did, 2))
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        vec = extract_optC(X_tr, n_cls)
        n_comp = len(vec) // 2
        rec = {'dataset_id': did}
        for j, v in enumerate(vec[:n_comp]):
            rec[f'dl_mean_{j}'] = float(v)
        for j, v in enumerate(vec[n_comp:]):
            rec[f'dl_var_{j}'] = float(v)
        rows_c.append(rec)
        done_c.add(did)
        elapsed = time.time() - t0
        print(f'[{i+1:3d}/{total}]  did={did}  n_comp={n_comp}  ({elapsed:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_c.append({'dataset_id': did})
        done_c.add(did)

    pd.DataFrame(rows_c).to_csv(CKPT_C, index=False)

optC_df = pd.DataFrame(rows_c)
print(f'\nOption C done. Shape: {optC_df.shape}')
optC_df.head(3)

Starting Option C fresh
[  1/51]  did=43924  n_comp=10  (1.6s)
[  2/51]  did=732  n_comp=8  (2.8s)
[  3/51]  did=1459  n_comp=20  (25.4s)
[  4/51]  did=4538  n_comp=10  (60.7s)
[  5/51]  did=694  n_comp=18  (1.1s)
[  6/51]  did=43958  n_comp=8  (18.0s)
[  7/51]  did=42544  n_comp=16  (3.2s)
[  8/51]  did=46944  n_comp=8  (38.5s)
[  9/51]  did=1446  n_comp=8  (5.8s)
[ 10/51]  did=45035  n_comp=8  (10.7s)
[ 11/51]  did=39  n_comp=16  (2.6s)
[ 12/51]  did=44033  n_comp=8  (11.5s)
[ 13/51]  did=1527  n_comp=10  (18.4s)
[ 14/51]  did=41842  n_comp=8  (27.5s)
[ 15/51]  did=44367  n_comp=8  (22.5s)
[ 16/51]  did=41168  n_comp=8  (57.9s)
[ 17/51]  did=44505  n_comp=14  (20.1s)
[ 18/51]  did=45060  n_comp=8  (9.9s)
[ 19/51]  did=1513  n_comp=10  (0.8s)
[ 20/51]  did=46543  n_comp=8  (32.0s)
[ 21/51]  did=41858  n_comp=8  (11.4s)
[ 22/51]  did=45028  n_comp=8  (15.3s)
[ 23/51]  did=734  n_comp=8  (27.8s)
[ 24/51]  did=46532  n_comp=8  (12.8s)
[ 25/51]  did=46842  n_comp=8  (1.9s)
[ 26/51]  did=4

,dataset_id,dl_mean_0,dl_mean_1,dl_mean_2,dl_mean_3,dl_mean_4,dl_mean_5,dl_mean_6,dl_mean_7,dl_mean_8,...,dl_var_10,dl_var_11,dl_var_12,dl_var_13,dl_var_14,dl_var_15,dl_var_16,dl_var_17,dl_var_18,dl_var_19
0,43924,-0.010341,0.021037,0.027774,-0.039934,-0.012901,-0.031685,0.002464,0.088558,0.068969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,732,-0.021890,-0.016181,-0.003673,0.026921,0.040506,-0.013600,0.024782,-0.043408,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1459,-0.027827,-0.055809,0.019554,-0.075757,0.036167,-0.027274,0.021300,0.049507,0.063801,...,0.057198,0.071317,0.0758,0.065389,0.104192,0.199399,0.087577,0.110342,0.068151,0.096363


## Merge & Save Final Tables

In [ ]:
# ── Join meta-features with LSE targets ───────────────────────────────────────

TARGET_COLS = ['dataset_id'] + LSE_COLS + ['best_method', 'gt_accuracy']
targets = lse_df[TARGET_COLS]

# Option A
df_a = targets.merge(optA_df, on='dataset_id', how='inner')
df_a.to_csv(OUT_A, index=False)
print(f'Option A saved → {OUT_A}  shape={df_a.shape}')

# Option B
df_b = targets.merge(optB_df, on='dataset_id', how='inner')
df_b.to_csv(OUT_B, index=False)
print(f'Option B saved → {OUT_B}  shape={df_b.shape}')

# Option C
df_c = targets.merge(optC_df, on='dataset_id', how='inner')
df_c.to_csv(OUT_C, index=False)
print(f'Option C saved → {OUT_C}  shape={df_c.shape}')

# Update main meta_training.csv with Option A features (main approach)
main_df = lse_df.merge(optA_df, on='dataset_id', how='left')
main_df.to_csv(LSE_CSV, index=False)
print(f'Main table updated → {LSE_CSV}  shape={main_df.shape}')

In [8]:
# ── Validation ────────────────────────────────────────────────────────────────

print('=== Option A feature summary ===')
mf_cols = [c for c in df_a.columns if c not in TARGET_COLS]
print(f'  {len(mf_cols)} meta-features: {mf_cols}')
print(df_a[mf_cols].describe().round(3).to_string())

print(f'\n=== NaN counts in Option A ===')
nan_counts = df_a[mf_cols].isna().sum()
print(nan_counts[nan_counts > 0] if nan_counts.any() else '  None — all features complete')

print(f'\n=== Option B shape: {df_b.shape} ===')
print(f'=== Option C shape: {df_c.shape} ===')

# Showcase exclusion sanity check
for name, df in [('A', df_a), ('B', df_b), ('C', df_c)]:
    leaked = set(df['dataset_id']) & SHOWCASE_IDS
    assert len(leaked) == 0, f'Showcase leak in Option {name}: {leaked}'

print('\nAll sanity checks passed.')
print('Phase 3 complete. Ready for Phase 4 (meta-learner training).')

=== Option A feature summary ===
  18 meta-features: ['n_instances', 'n_features', 'n_classes', 'skewness_mean', 'kurtosis_mean', 'mean_abs_pearson', 'class_entropy', 'imbalance_ratio', 'hopkins', 'intrinsic_dim_ratio', 'pca_var_pc1', 'inter_intra_ratio', 'silhouette_true', 'davies_bouldin_true', 'knn1_accuracy', 'decision_stump_accuracy', 'feature_sparsity', 'cv_mean']
       n_instances  n_features  n_classes  skewness_mean  kurtosis_mean  mean_abs_pearson  class_entropy  imbalance_ratio  hopkins  intrinsic_dim_ratio  pca_var_pc1  inter_intra_ratio  silhouette_true  davies_bouldin_true  knn1_accuracy  decision_stump_accuracy  feature_sparsity       cv_mean
count       51.000      51.000     51.000         50.000         50.000            50.000         51.000           51.000   51.000               51.000       51.000             51.000           51.000               51.000         51.000                   51.000            51.000  5.100000e+01
mean     11565.706      19.922      3.4

In [ ]:
# ── Quick look: which features correlate most with best_method diversity ───────

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Compute per-dataset LSE variance across methods (meta-learnability signal)
df_a['lse_std'] = df_a[LSE_COLS].std(axis=1)

corrs = df_a[mf_cols + ['lse_std']].corr()['lse_std'].drop('lse_std').abs().sort_values(ascending=False)

print('=== Feature correlation with LSE variance (meta-learnability signal) ===')
print(corrs.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
corrs.plot.barh(ax=ax)
ax.set_xlabel('|Pearson r| with LSE std across methods')
ax.set_title('Option A: Feature relevance for meta-learning')
plt.tight_layout()
fig_path = os.path.join(ROOT, 'outputs', 'figures', 'optA_feature_relevance.png')
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
fig.savefig(fig_path, dpi=120)
plt.close()
print(f'\nFigure saved → {fig_path}')